In [ ]:
"""
Callset benchmarking and comparison.
"""

import copy
import importlib
import re
import pickle
import warnings
from pathlib import Path
from collections import defaultdict
import time

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
import pysam
import seaborn as sns
from scipy.spatial import cKDTree
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import os
from typing import Union, Tuple, Optional, List, Any, Literal, Dict


workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2"
os.chdir(workdir)
os.getcwd()

import svensemble.vcfparse as vcfp # noqa: E402 (flake8)
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.benchmark as benchmark
import svensemble._caller_params as caller_params
import svensemble._sample_params as sample_params
import svensemble.chunk_match as chunk_match
import svensemble.matched_breakpoints as matchedbps
import svensemble.utils as sv_utils

importlib.reload(vcfp)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(benchmark)
importlib.reload(caller_params)
importlib.reload(sample_params)
importlib.reload(chunk_match)
importlib.reload(matchedbps)
importlib.reload(sv_utils)

pd.set_option('display.max_columns', None)

long_val = True
model_name = 'unbiased'


In [ ]:

training_sets =  {
    'gridss': ['gridss'],
    'manta_candidate': ['manta_candidate'],
    'manta_diploid': ['manta_diploid'],
    'gridss+manta_candidate': ['gridss', 'manta_candidate'],
    'gridss+manta_diploid': ['gridss', 'manta_diploid']
}
training_sets = caller_params.training_set_presets['default']

sample_names = ['NA12878_30x', 'CHM13', 'NA12878', 'HG002']
# sample_names = ['NA12878_30x', 'CHM13', 'NA12878']
# sample_names = ['NA12878_30x']

svens2_model_name = 'unbiased'
long_suffix = '_long' if long_val else ''

val_vcf_paths = {
    'kosugi': '/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/Kosugi_validation/NA12878_DGV-2016_LR-assembly.hg38.bed',
    'chm13_val': '/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/CHM13_validation/supporting_variants_for_nstd137.csv',
    'hg002_val': '/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/data/HG002_validation/HG002_validation.vcf.gz'
}


# callset_qual_metrics = {
#     'gridss': 'QUAL',
#     'manta_candidate': 'INFO_TOTAL_PAIR_COUNT',
#     'manta_diploid': 'QUAL',
#     'delly': 'QUAL'
# }

samples = {}
for sample_name in sample_names:
    samples.update(sv_utils.create_config_dict(sample_name, training_sets=training_sets, val_set='use_name'))

In [ ]:
featureimportance_config_path = '/home/users/allstaff/shikov.v/projects/Papenfuss_SVEnsemble2/model/figs/featureimportance/svensemble2_importances_config.csv'
sv_utils.create_samples_config_table(samples).to_csv(featureimportance_config_path, index=False)

In [ ]:
# Val set preprocessing
if 'kosugi' in val_vcf_paths.keys():
    benchmark.preprocess_kosugi_val_set(sample=samples['NA12878'], vcf_path=val_vcf_paths['kosugi'])
if 'chm13_val' in val_vcf_paths.keys():
    benchmark.preprocess_chm13_val_set(sample=samples['CHM13'], vcf_path=val_vcf_paths['chm13_val'])
if 'hg002_val' in val_vcf_paths.keys():
    benchmark.preprocess_hg002_val_set(sample=samples['HG002'], vcf_path=val_vcf_paths['hg002_val'])

In [ ]:
for sample_name in samples.keys():
    samples[sample_name].update({'matched_bps': {training_set_name: None for training_set_name in samples[sample_name]['training_sets']}})
    for training_set_name in samples[sample_name]['training_sets']:
        samples[sample_name]['matched_bps'][training_set_name] = pd.read_csv(samples[sample_name]['sample_dir'] / 'sv_calling' / 'svensemble2' / f'matched_bps_{training_set_name}.csv', low_memory=False)
    samples[sample_name]['pred_dicts'] = pickle.load(open(samples[sample_name]['sample_dir'] / 'sv_calling' / 'svensemble2' / 'unbiased_dict.pkl', 'rb'))
    # Predictions have ID columns as index, change them to usual columns
    for training_set, pred_dict in samples[sample_name]['pred_dicts'].items():
        pred_dict['y'].reset_index(inplace=True, drop=False)
        pred_dict['y'].drop(columns=['gnomad_INFO_EVENT', 0], inplace=True)
        pred_dict['y'].rename(columns={1: f'{training_set}_{svens2_model_name}_pred'}, inplace=True)
    # Ensure all column names in pred_dicts prediction dataframes are strings, not integers
    # for training_set, pred_dict in samples[sample_name]['pred_dicts'].items():
    #     pred_dict['y'].columns = pred_dict['y'].columns.astype(str)


In [ ]:
# def merge_predictions(matched_bps: pd.DataFrame, preds: pd.DataFrame, training_set_name: str, svens2_model_name: str):
#     # First, check if matched_bps contains INFO_EVENT columns from preds
#     matched_bps_info_event_cols = [col for col in matched_bps.columns if 'INFO_EVENT' in col]
#     pred_info_event_cols = [col for col in preds.columns if 'INFO_EVENT' in col]
#     if set(matched_bps_info_event_cols).intersection(set(pred_info_event_cols)) != set(pred_info_event_cols):
#         raise ValueError(f"INFO_EVENT columns in matched_bps and pred_dict do not match: {matched_bps_info_event_cols} != {pred_info_event_cols}, cannot create matches")

#     # Pick the column with svensemble2 positive predictions from preds and rename it
#     # The required column is always the last one
#     svens2_pred_col_name = f"{training_set_name}_{svens2_model_name}_pred"
#     preds = preds[pred_info_event_cols + [preds.columns[-1]]]
#     preds = preds.rename({preds.columns[-1]: svens2_pred_col_name}, axis=1)

#     # Left-merge preds with matched_bps on the INFO_EVENT columns
#     matched_bps = matched_bps.merge(preds, on=pred_info_event_cols, how='left')
#     return matched_bps

In [ ]:
for sample_name in samples.keys():
    for training_set_name, training_set_callsets in samples[sample_name]['training_sets'].items():
        if 'gridss' in training_set_callsets:
            samples[sample_name]['matched_bps'][training_set_name].loc[:, 'gridss_FILTER_PASS'] = np.all(samples[sample_name]['matched_bps'][training_set_name][['gridss_FILTER_PASS_bnd1', 'gridss_FILTER_PASS_bnd2']].astype(bool), axis=1)
        if 'manta_candidate' in training_set_callsets:
            samples[sample_name]['matched_bps'][training_set_name].loc[:, 'manta_candidate_INFO_TOTAL_PAIR_COUNT'] = \
                samples[sample_name]['matched_bps'][training_set_name]['manta_candidate_INFO_UPSTREAM_PAIR_COUNT'] + \
                samples[sample_name]['matched_bps'][training_set_name]['manta_candidate_INFO_DOWNSTREAM_PAIR_COUNT'] + \
                samples[sample_name]['matched_bps'][training_set_name]['manta_candidate_INFO_PAIR_COUNT']


In [ ]:
# Set '[training_set]_unbiased_pred' column value for matched breakpoints to the predicted value. Put NA if there is no prediction

for sample_name in samples:
    for training_set, training_callsets in training_sets.items():
        model_name = 'unbiased'

        samples[sample_name]['matched_bps'][training_set] = samples[sample_name]['matched_bps'][training_set].merge(
            samples[sample_name]['pred_dicts'][training_set]['y'],
            on=[f'{callset}_INFO_EVENT' for callset in training_callsets]
        )


In [ ]:
def get_svensemble2_calls(
    matchedbps_df: pd.DataFrame,
    pred_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Summarizes SVEnsemble2 calls by merging matched breakpoint and prediction DataFrames,
    computing combined insert sizes, generating unique SVEnsemble2 IDs, and formatting results.

    Args:
        matchedbps_df (pd.DataFrame): DataFrame of matched breakpoints.
        callsets (list[str]): List of callset names included in this training set.
        pred_df (pd.DataFrame): DataFrame containing model predictions and INFO_EVENT keys.
        model_name (str): String indicating the model/prediction column name to carry forward (e.g., "unbiased").

    Returns:
        pd.DataFrame: Unified DataFrame of SVEnsemble2 summary calls with consensus coordinates, insert size, unique ID, and predictions.
    """
    # Create a working copy of the matched breakpoints to avoid modifying the input DataFrame
    matchedbps_df_copy = matchedbps_df.copy()

    # Infer callset list based on prefixes of INFO_EVENT columns in pred_df
    callsets = [col.split('_')[0] for col in pred_df.columns if 'INFO_EVENT' in col]

    # Merge the predictions into the matched breakpoints data using the INFO_EVENT columns as keys
    matchedbps_df_copy = matchedbps_df_copy.merge(
        pred_df,
        on=[f'{callset}_INFO_EVENT' for callset in callsets]
    )

    # Compute the combined (aggregate/max) insert size across all callsets and store in a new column
    matchedbps_df_copy['svensemble2_insert_size'] = matchedbps.get_combined_insert_size(
        matchedbps_df_copy, callsets
    )

    # Generate a unique identifier for each SV by concatenating INFO_EVENT fields from all callsets
    # This is useful for matching SVs across different sets and preserving provenance
    matchedbps_df_copy.loc[:, 'INFO_EVENT'] = matchedbps.generate_svensemble2_id(
        matchedbps_df_copy, callsets
    )

    # Infer the training_set_name and model_name from the single prediction column in pred_df
    # Expect column like "{training_set_name}_{model_name}_pred"
    prediction_cols = [col for col in pred_df.columns if col.endswith('_pred')]
    if len(prediction_cols) != 1:
        raise ValueError("There should be exactly one prediction column in pred_df ending with '_pred'")
    pred_col = prediction_cols[0]
    # Split out training_set_name and model_name
    name_parts = pred_col.rsplit('_', 2)
    if len(name_parts) == 3:
        training_set_name, model_name, _ = name_parts
    else:
        raise ValueError(f"Prediction column name does not match expected pattern: '{pred_col}'")


    # Convert to a SVEnsemble2 "bps" format, retaining coordinate/ALT consensus, insert size, unique ID, and predictions column
    return matchedbps.matchedbps_to_bps(
        matchedbps_df_copy,
        callsets,
        extra_cols=[
            'svensemble2_insert_size',
            'INFO_EVENT',
            f"{training_set_name}_{model_name}_pred"
        ]
    )

In [ ]:
for sample_name, sample in samples.items():
    samples[sample_name].update({'svensemble2_calls': {}})
    for training_set_name, training_callsets in sample['training_sets'].items():
        matched_bps = sample['matched_bps'][training_set_name].copy()
        matched_bps['svensemble2_insert_size'] = matchedbps.get_combined_insert_size(matched_bps, training_callsets)
        #create a unique id for each sv
        matched_bps.loc[:, 'INFO_EVENT'] = matchedbps.generate_svensemble2_ids(matched_bps, training_callsets)

        samples[sample_name]['svensemble2_calls'].update(
            {
                training_set_name: matchedbps.matchedbps_to_bps(matched_bps, training_callsets, extra_cols=['svensemble2_insert_size', 'INFO_EVENT'] + [f'{training_set_name}_{model_name}_pred'])
            }
        )


In [ ]:
pickle.dump(samples, open('model/saved/samples.pkl', 'wb'))

In [ ]:
samples = pickle.load(open('model/saved/samples.pkl', 'rb'))

In [ ]:
samples['NA12878']['callsets']

In [ ]:
verbose = True

# Here we create a validation dictinary containing
# Validation dictionary is a nested dictionary, with upper level keys being the sample names
# For each sample, a dictionary exists with for each callset as well as for each svensemble2 prediction

val_dict = defaultdict(dict)

for sample_name, sample in samples.items():
    if sample['val_set'] is None:
        continue

    val_set = pd.read_csv(sample['val_set']['path'])

    # First, handle original callsets that will be compared to svensemble2 predictions
    for callset in sample['callsets']:
        if verbose:
            print(f'Processing {callset} callset for {sample_name}')

        # Recreate the original callset bps df by picking only the rows that contain the corresponding INFO_EVENT
        # Then remove all columns without the callset prefix
        callset_bps = pd.read_csv(sample['callsets'][callset]['svensemble_dir'] / f'{callset}_bps.csv', low_memory=False)

        # Caller-specific operations go here
        if callset == 'manta_candidate':
            callset_bps.loc[:, 'INFO_TOTAL_PAIR_COUNT'] = \
                callset_bps['INFO_UPSTREAM_PAIR_COUNT'] + \
                callset_bps['INFO_DOWNSTREAM_PAIR_COUNT'] + \
                callset_bps['INFO_PAIR_COUNT']

        matched_ids = chunk_match.inner_join_bps(
            bp_df_dict={
                callset: callset_bps,
                sample['val_set']['name']: val_set
            },
            max_dist=200,
            ids_only=True,
            ignore_size=True
        )
        callset_bps.loc[:, 'is_in_val'] = callset_bps['INFO_EVENT'].isin(matched_ids[f'{callset}_INFO_EVENT'])
        cols_to_keep = ['INFO_EVENT', f'{sample['callsets'][callset]['params']['qual_metric']}', 'is_in_val']
        if 'FILTER_PASS' in callset_bps.columns:
            cols_to_keep.append('FILTER_PASS')
        val_dict[sample_name][callset] = callset_bps.loc[callset_bps['INFO_EVENT'].notna(), cols_to_keep].copy()

        if verbose:
            match_count = val_dict[sample_name][callset]['is_in_val'].sum()
            print(f'{callset} callset for {sample_name} has {match_count} rows matching the validation set')

    # Then, handle svensemble2 predictions

    for training_set, training_callsets in samples[sample_name]['training_sets'].items():

        if verbose:
            print(f'Processing SVEnsemble2 predictions with {training_set} for {sample_name}')

        svens_calls = sample['svensemble2_calls'][training_set]

        matched_ids = chunk_match.inner_join_bps(
            bp_df_dict={
                'svensemble2': svens_calls,
                sample['val_set']['name']: val_set
            },
            max_dist=200,
            ids_only=True,
            ignore_size=True
        )

        svens_calls.loc[:, 'is_in_val'] = svens_calls['INFO_EVENT'].isin(matched_ids['svensemble2_INFO_EVENT'])

        val_df = svens_calls[
            [
                'INFO_EVENT',
                f'{training_set}_{model_name}_pred',
                'is_in_val'
            ]
        ]
        val_df = val_df[val_df[f'{training_set}_{model_name}_pred'].notna()]

        val_dict[sample_name][f'svensemble2_{training_set}'] = val_df.copy()

        if verbose:
            match_count = val_dict[sample_name][f'svensemble2_{training_set}']['is_in_val'].sum()
            print(f'SVEnsemble2  predictions with {training_set} for {sample_name} has {match_count} rows matching the validation set')


In [ ]:
cols_to_keep

In [ ]:
sample['callsets']['manta_candidate']['qual_metric']

In [ ]:
pickle.dump(val_dict, open('model/saved/val_dict.pkl', 'wb'))

In [ ]:
test_sample_name = 'NA12878'
test_callset_name = 'gridss'
test_training_set_name = 'gridss+manta_diploid'

test_svens_calls = samples[test_sample_name]['svensemble2_calls'][test_training_set_name]
test_callset_bps = samples[test_sample_name]['matched_bps'][test_training_set_name]

test_svens_calls_cipos_cols = [col for col in test_svens_calls.columns if 'INFO_CIPOS' in col]
test_callset_cipos_cols = [col for col in test_callset_bps.columns if 'INFO_CIPOS' in col]

test_svens_calls_max_cipos_sizes = test_svens_calls[test_svens_calls_cipos_cols].abs().max(axis=1)
test_callset_bps_max_cipos_sizes = test_callset_bps[test_callset_cipos_cols].abs().max(axis=1)

(test_svens_calls_max_cipos_sizes > 300).value_counts()



In [ ]:
(test_callset_bps_max_cipos_sizes > 300).value_counts()

In [ ]:
def plot_cipos_boxplot(df, ax=None, type='box'):
    plot_type_dispatch = {
        'box': sns.boxplot,
        'violin': sns.violinplot
    }
    try:
        plot_func = plot_type_dispatch[type]
    except KeyError:
        raise ValueError(f"Invalid plot type: {type}. Must be one of: {', '.join(plot_type_dispatch.keys())}")
    # Extract the relevant DataFrame
    cipos_cols = [col for col in df.columns if 'INFO_CIPOS' in col]
    df = df[cipos_cols + ['is_in_val']]

    # Take absolute values and prevent log(0). Log-transform the values
    df = df.abs()
    for df_col in df.columns[:-1]:
        df.loc[df[df_col] == 0, df_col] = 1
        df.loc[:, df_col] = np.log(df[df_col])

    df_melted = df.melt(id_vars=['is_in_val'], value_vars=df.columns[:-1])

    # Create a boxplot grouped by 'is_in_val'
    fig, ax = plt.subplots(figsize=(10, 6))

    plot_func(x='variable', y='value', hue='is_in_val', data=df_melted, ax=ax)


In [ ]:
plot_cipos_boxplot(samples['NA12878']['svensemble2_calls']['gridss'], type='violin')

In [ ]:
plot_cipos_boxplot(samples['NA12878']['svensemble2_calls']['gridss+manta_diploid'], type='violin')

In [ ]:
plot_cipos_boxplot(samples['NA12878']['svensemble2_calls']['gridss+manta_diploid+delly'], type='violin')

In [ ]:
samples['NA12878']['matched_bps']['gridss+manta_diploid']

In [ ]:
val_dict['NA12878']['gridss']

In [ ]:
callset_name = 'gridss'
plot_cipos_boxplot(samples['NA12878']['matched_bps'][f'{callset_name}'].merge(val_dict['NA12878'][callset_name].rename(columns={'INFO_EVENT': f'{callset_name}_INFO_EVENT'}), on=f'{callset_name}_INFO_EVENT', how='left'))

In [ ]:
callset_name = 'delly'
plot_cipos_boxplot(samples['NA12878']['matched_bps'][f'{callset_name}'].merge(val_dict['NA12878'][callset_name].rename(columns={'INFO_EVENT': f'{callset_name}_INFO_EVENT'}), on=f'{callset_name}_INFO_EVENT', how='left'))

In [ ]:
callset_name = 'manta_diploid'
plot_cipos_boxplot(samples['NA12878']['matched_bps'][f'{callset_name}'].merge(val_dict['NA12878'][callset_name].rename(columns={'INFO_EVENT': f'{callset_name}_INFO_EVENT'}), on=f'{callset_name}_INFO_EVENT', how='left'))

In [ ]:
plot_cipos_boxplot(samples['NA12878']['svensemble2_calls']['gridss+manta_diploid'])

In [ ]:
samples['NA12878']['svensemble2_calls']['gridss+manta_diploid'][['INFO_CIPOS_r_bnd1', 'INFO_CIPOS_l_bnd2']].describe()

In [ ]:
samples['NA12878']['matched_bps']['gridss+manta_diploid'][['gridss_INFO_CIPOS_r_bnd1', 'gridss_INFO_CIPOS_l_bnd2']].describe()

In [ ]:
samples['NA12878']['matched_bps']['gridss+manta_diploid'][['manta_diploid_INFO_CIPOS_r_bnd1', 'manta_diploid_INFO_CIPOS_l_bnd2']].describe()

In [ ]:
if 'val_dict.pkl' in os.listdir():
    pickle.dump(val_dict, open('val_dict.pkl', 'wb'))

In [ ]:
val_dict = pickle.load(open('val_dict.pkl', 'rb'))

In [ ]:
val_dict['NA12878'].keys()

In [ ]:
for sample_name in val_dict.keys():
    for pred_set in val_dict[sample_name].keys():
        print(f'Sample: {sample_name},\nPred set: {pred_set},\nCounts: {val_dict[sample_name][pred_set]["is_in_val"].value_counts()}\n\n')

In [ ]:
for sample_name in samples.keys():
    benchmark.plot_reverse_roc_curves(val_dict[sample_name], filter_pass=False, title=sample_name)

In [ ]:
benchmark.plot_roc_curves(val_dict['NA12878_30x'], filter_pass=False, title='NA12878_30x')

In [ ]:
benchmark.plot_roc_curves({val_set_name: val_set for val_set_name, val_set in val_dict['NA12878_30x'].items() if val_set_name in ['manta_diploid', 'svensemble2_manta_diploid']}, filter_pass=False, title='NA12878_30x')

In [ ]:
benchmark.plot_roc_curves(val_dict['NA12878'], filter_pass=True, title='NA12878')

In [ ]:
benchmark.plot_roc_curves(val_dict['CHM13'], filter_pass=True, title='CHM13')

In [ ]:
benchmark.plot_roc_curves(val_dict['HG002'], ax=None, use_counts=True, title='HG002')

In [ ]:
benchmark.plot_precision_recall_curves(val_dict['HG002'])

In [ ]:
samples['NA12878']['pred_dicts']

In [ ]:
import sklearn.model_selection as skm

verbose = True
sample_config = samples['NA12878']
reference_sample_config = sv_utils.create_config_dict('gnomad', is_reference=True)['gnomad']
training_set_name = 'gridss+manta_diploid'



In [ ]:
sample_name = 'NA12878'
training_set_name = 'gridss+manta_diploid'

svensemble2_model = samples[sample_name]['pred_dicts'][training_set_name]['RF_fit']
svensemble2_splits = load_svensemble2_train_test_splits(samples[sample_name], sv_utils.create_config_dict('gnomad', is_reference=True)['gnomad'], training_set_name, verbose=True)
svensemble2_X_val = svensemble2_splits['X_val']
svensemble2_y_val = svensemble2_splits['y_val']



In [ ]:
svensemble2_importances = svensemble2_premutation_importance(svensemble2_model, svensemble2_X_val, svensemble2_y_val, n_repeats=10, random_state=42)

In [ ]:
svensemble2_importances

In [ ]:
fig, ax = plt.subplots()
svensemble2_importances.sort_values(ascending=False).plot(ax=ax, kind='barh')
ax.set_title("Mean accuracy decrease")
ax.set_ylabel("Feature importances using permutation on full model")

fig.tight_layout()
fig.set_size_inches((5,10))
plt.show()
# plt.show()
# plt.savefig('figs/featureimportance_perm.enoto.png')